<div align="center">

# Atividade — Regressão Logística vs. Árvore de Decisão

**Instituto Federal de Educação, Ciência e Tecnologia de São Paulo — IFSP**

**Disciplina:** Introdução ao Aprendizado de Máquina (CMPINAM)  
**Curso:** Análise e Desenvolvimento de Sistemas  
**Professor:** Prof. Eveton Meyer  

</div>

---

**Aluno(a)(s):** ______________________________________________  
**Prontuário(s):** ____________________________________________  
**Data:** ____ / ____ / ______

---

## Objetivos da atividade

Ao final desta atividade, você deverá ser capaz de:

- preparar os dados de forma adequada para diferentes classificadores;
- treinar modelos de **Regressão Logística** e **Árvore de Decisão**;
- comparar os modelos usando **matriz de confusão, acurácia, precisão, recall e F1-score**;
- interpretar os resultados no contexto de **risco de inadimplência**;
- reconhecer por que o **desbalanceamento de classes** exige atenção além da acurácia.

## 1. Dataset: Default of Credit Card Clients

O conjunto de dados **Default of Credit Card Clients** contém informações de **30.000 clientes de cartão de crédito** e tem como objetivo prever a ocorrência de inadimplência no mês seguinte.

- **Fonte:** UCI Machine Learning Repository
- **Tarefa:** classificação binária
- **Variável-alvo:** `default`
  - `0` → cliente não inadimplente no mês seguinte
  - `1` → cliente inadimplente no mês seguinte

Entre as variáveis preditoras estão informações de limite de crédito, idade, características demográficas, histórico de pagamentos, valores de faturas e valores pagos.

Nesta atividade, o foco não é explorar todas as particularidades do dataset, mas utilizá-lo para comparar dois classificadores e interpretar suas métricas.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

## 2. Carregamento dos dados

O arquivo `default_of_credit_card_clients.xls` deve estar no mesmo diretório deste notebook.

Após o carregamento:

1. a variável-alvo será renomeada para `default`;
2. a coluna `ID` será removida, pois funciona apenas como identificador.

In [2]:
df = pd.read_excel("default_of_credit_card_clients.xls", header=1)

df.rename(
    columns={"default payment next month": "default"},
    inplace=True
)

df.drop(columns=["ID"], inplace=True)

print(f"Dimensões do dataset: {df.shape}")
df.head()

Dimensões do dataset: (30000, 24)


,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,50000,2,2,1,37,0,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,50000,1,2,1,57,-1,0,-1,0,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 24 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   LIMIT_BAL  30000 non-null  int64
 1   SEX        30000 non-null  int64
 2   EDUCATION  30000 non-null  int64
 3   MARRIAGE   30000 non-null  int64
 4   AGE        30000 non-null  int64
 5   PAY_0      30000 non-null  int64
 6   PAY_2      30000 non-null  int64
 7   PAY_3      30000 non-null  int64
 8   PAY_4      30000 non-null  int64
 9   PAY_5      30000 non-null  int64
 10  PAY_6      30000 non-null  int64
 11  BILL_AMT1  30000 non-null  int64
 12  BILL_AMT2  30000 non-null  int64
 13  BILL_AMT3  30000 non-null  int64
 14  BILL_AMT4  30000 non-null  int64
 15  BILL_AMT5  30000 non-null  int64
 16  BILL_AMT6  30000 non-null  int64
 17  PAY_AMT1   30000 non-null  int64
 18  PAY_AMT2   30000 non-null  int64
 19  PAY_AMT3   30000 non-null  int64
 20  PAY_AMT4   30000 non-null  int64
 21  PAY_AMT5   3

## 3. Verificação inicial

Antes do treinamento, vamos observar a distribuição da variável-alvo.

Esse passo é importante porque, em problemas de classificação, uma classe pode aparecer com muito mais frequência que a outra. Quando isso ocorre, a **acurácia isoladamente pode fornecer uma visão incompleta do desempenho do modelo**.

In [4]:
distribuicao = (
    df["default"]
    .value_counts()
    .sort_index()
    .rename_axis("classe")
    .to_frame("quantidade")
)

distribuicao["proporcao"] = (
    df["default"]
    .value_counts(normalize=True)
    .sort_index()
)

distribuicao

,quantidade,proporcao
classe,,
0,23364,0.7788
1,6636,0.2212


## 4. Divisão em treino e teste

Vamos separar os dados em:

- **75% para treinamento**;
- **25% para teste**.

Será utilizada **estratificação pela variável `default`**, preservando aproximadamente a mesma proporção das classes nos dois conjuntos.

O `random_state=42` permite reproduzir a mesma divisão em diferentes execuções.

In [5]:
X = df.drop(columns=["default"]).copy()
y = df["default"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("=== Divisão dos dados ===")
print(f"Treino: {X_train.shape[0]} amostras")
print(f"Teste : {X_test.shape[0]} amostras")

print("\n=== Proporção das classes no treino ===")
print(y_train.value_counts(normalize=True).sort_index().round(3))

print("\n=== Proporção das classes no teste ===")
print(y_test.value_counts(normalize=True).sort_index().round(3))

=== Divisão dos dados ===
Treino: 22500 amostras
Teste : 7500 amostras

=== Proporção das classes no treino ===
default
0    0.779
1    0.221
Name: proportion, dtype: float64

=== Proporção das classes no teste ===
default
0    0.779
1    0.221
Name: proportion, dtype: float64


## 5. Pré-processamento

Nem todas as colunas devem ser tratadas da mesma maneira.

### Variáveis categóricas

Nesta atividade, trataremos como categóricas:

- `SEX`
- `EDUCATION`
- `MARRIAGE`

Embora estejam armazenadas como números inteiros, seus valores representam **categorias**. Por isso, serão transformadas com `OneHotEncoder`.

> **Importante:** o tipo `int64` de uma coluna não significa, por si só, que ela deve ser interpretada como uma grandeza numérica contínua.

### Demais variáveis

As demais colunas serão mantidas como numéricas. Isso inclui os atributos `PAY_*`, que serão tratados, para fins desta atividade introdutória, como variáveis ordinais numéricas.

Na **Regressão Logística**, as variáveis numéricas serão padronizadas com `StandardScaler`.

Na **Árvore de Decisão**, a padronização não é necessária, pois suas divisões não dependem da escala das variáveis.

In [6]:
categorical_features = ["SEX", "EDUCATION", "MARRIAGE"]

numeric_features = [
    col for col in X.columns
    if col not in categorical_features
]

preprocessor_log = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "num",
            StandardScaler(),
            numeric_features
        )
    ]
)

preprocessor_tree = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "num",
            "passthrough",
            numeric_features
        )
    ]
)

print("Variáveis categóricas:", categorical_features)
print("Quantidade de variáveis numéricas/ordinais:", len(numeric_features))

Variáveis categóricas: ['SEX', 'EDUCATION', 'MARRIAGE']
Quantidade de variáveis numéricas/ordinais: 20


## 6. Treinamento dos modelos

Serão comparados:

### Regressão Logística
Modelo linear que estima a probabilidade de uma observação pertencer à classe `1`.

### Árvore de Decisão
Modelo baseado em divisões sucessivas dos dados.

Nesta primeira comparação, a árvore será treinada **sem limitar sua profundidade**. Observe os resultados com atenção: uma árvore muito complexa pode se ajustar excessivamente aos dados de treinamento.

In [7]:
log_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor_log),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                random_state=42
            )
        )
    ]
)

tree_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor_tree),
        (
            "classifier",
            DecisionTreeClassifier(
                random_state=42
            )
        )
    ]
)

log_model.fit(X_train, y_train)
tree_model.fit(X_train, y_train)

print("Modelos treinados com sucesso.")

Modelos treinados com sucesso.


## 7. Avaliação dos modelos

Nesta primeira avaliação, será utilizado o **limiar padrão de 0,5** dos classificadores.

Isso significa que, de maneira simplificada, o modelo atribui uma observação à classe `1` quando sua decisão indica que ela deve ser classificada como inadimplente segundo esse limiar.

As principais métricas serão:

- **Matriz de confusão:** apresenta verdadeiros negativos, falsos positivos, falsos negativos e verdadeiros positivos.
- **Acurácia:** proporção total de classificações corretas.
- **Precisão da classe 1:** entre os clientes classificados como inadimplentes, quantos realmente eram inadimplentes.
- **Recall da classe 1:** entre os clientes realmente inadimplentes, quantos foram identificados pelo modelo.
- **F1-score da classe 1:** média harmônica entre precisão e recall.

No contexto desta atividade, dê atenção especial aos erros envolvendo a **classe `1`**.

In [8]:
y_pred_log = log_model.predict(X_test)
y_pred_tree = tree_model.predict(X_test)

In [9]:
def avaliar_modelo(nome, y_true, y_pred):
    print(f"=== {nome} ===")
    print("\nMatriz de confusão:")
    print(confusion_matrix(y_true, y_pred))

    print("\nRelatório de classificação:")
    print(classification_report(y_true, y_pred, digits=3))

    print(f"Acurácia: {accuracy_score(y_true, y_pred):.3f}")
    print("-" * 60)


avaliar_modelo(
    "Regressão Logística",
    y_test,
    y_pred_log
)

avaliar_modelo(
    "Árvore de Decisão",
    y_test,
    y_pred_tree
)

=== Regressão Logística ===

Matriz de confusão:
[[5670  171]
 [1252  407]]

Relatório de classificação:
              precision    recall  f1-score   support

           0      0.819     0.971     0.889      5841
           1      0.704     0.245     0.364      1659

    accuracy                          0.810      7500
   macro avg      0.762     0.608     0.626      7500
weighted avg      0.794     0.810     0.772      7500

Acurácia: 0.810
------------------------------------------------------------
=== Árvore de Decisão ===

Matriz de confusão:
[[4748 1093]
 [1023  636]]

Relatório de classificação:
              precision    recall  f1-score   support

           0      0.823     0.813     0.818      5841
           1      0.368     0.383     0.375      1659

    accuracy                          0.718      7500
   macro avg      0.595     0.598     0.597      7500
weighted avg      0.722     0.718     0.720      7500

Acurácia: 0.718
---------------------------------------------

## 8. Comparação direta

A tabela abaixo resume as métricas mais importantes para a comparação.

Além dos dois modelos, será exibida a **acurácia de referência da classe majoritária**: o valor que seria obtido por uma estratégia ingênua que previsse sempre a classe mais frequente no conjunto de teste.

Essa referência ajuda a interpretar se uma acurácia aparentemente alta realmente representa um bom desempenho.

In [10]:
def metricas_classe_1(y_true, y_pred):
    return {
        "Acurácia": accuracy_score(y_true, y_pred),
        "Precisão (classe 1)": precision_score(y_true, y_pred, zero_division=0),
        "Recall (classe 1)": recall_score(y_true, y_pred, zero_division=0),
        "F1-score (classe 1)": f1_score(y_true, y_pred, zero_division=0),
    }


comparacao = pd.DataFrame(
    {
        "Regressão Logística": metricas_classe_1(
            y_test,
            y_pred_log
        ),
        "Árvore de Decisão": metricas_classe_1(
            y_test,
            y_pred_tree
        ),
    }
).T

baseline_majoritaria = y_test.value_counts(normalize=True).max()

print(
    f"Acurácia de referência da classe majoritária: "
    f"{baseline_majoritaria:.3f}\n"
)

comparacao.round(3)

Acurácia de referência da classe majoritária: 0.779



,Acurácia,Precisão (classe 1),Recall (classe 1),F1-score (classe 1)
Regressão Logística,0.810,0.704,0.245,0.364
Árvore de Decisão,0.718,0.368,0.383,0.375


# 9. Análise e discussão

Após executar todas as células anteriores, responda às questões **com suas próprias palavras**.

Não basta reproduzir os números. Explique **o que os resultados significam no contexto do problema**.

---

## Questão 1 — Comparação geral dos modelos

Qual modelo apresentou **maior acurácia no conjunto de teste**?

Na sua resposta:

- informe os valores encontrados;
- compare também com a acurácia de referência da classe majoritária;
- utilize a matriz de confusão para explicar os principais acertos e erros;
- diga se a diferença entre os dois modelos parece relevante.

---

## Questão 2 — Classe mais difícil de identificar

Qual classe foi mais difícil de classificar: clientes **não inadimplentes (`0`)** ou **inadimplentes (`1`)**?

Justifique utilizando:

- precisão;
- recall;
- F1-score;
- falsos positivos;
- falsos negativos.

Explique também como o desbalanceamento observado no dataset pode influenciar esse resultado.

---

## Questão 3 — Qual métrica deve receber maior atenção?

Considere que o objetivo seja **identificar clientes com risco de inadimplência**.

Quais métricas deveriam receber maior atenção na escolha entre os modelos?

Em sua resposta:

- explique por que a acurácia, isoladamente, pode ser insuficiente;
- discuta o significado do **recall da classe `1`**;
- discuta o significado da **precisão da classe `1`**;
- considere o custo de um **falso negativo** e de um **falso positivo** nesse contexto;
- apresente sua conclusão sobre qual modelo você escolheria com base nos resultados obtidos.

---

## Questão 4 — Complexidade da Árvore de Decisão

A Árvore de Decisão utilizada nesta atividade foi criada sem uma limitação explícita de profundidade.

Explique:

- o que pode acontecer quando uma árvore cresce excessivamente;
- o que significa **overfitting** nesse contexto;
- cite pelo menos dois hiperparâmetros que poderiam ser utilizados para controlar a complexidade da árvore.

---

## Questão 5 — Estratégias para lidar com o desbalanceamento

Pesquise **pelo menos três estratégias** que poderiam ser utilizadas para lidar com o desbalanceamento entre as classes.

Para cada estratégia:

1. informe seu nome;
2. explique resumidamente o que ela modifica no processo de treinamento ou decisão;
3. indique qual impacto ela poderia ter sobre a identificação da classe `1`.

Você pode investigar, por exemplo:

- pesos diferentes para as classes;
- técnicas de sobreamostragem;
- técnicas de subamostragem;
- alteração do limiar de decisão.

> O objetivo não é apenas listar nomes. É necessário explicar a lógica de cada estratégia.

---

## Entrega

Entregue este notebook com:

- todas as células executadas;
- os resultados preservados;
- as respostas das cinco questões preenchidas em células Markdown.

As respostas devem demonstrar interpretação e capacidade de relacionar as métricas ao problema de inadimplência.